In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 12
CUDA device: NVIDIA GeForce RTX 5070 Ti


# Kvasir-VQA x1 — BLIP VQA fine-tuning

Fine-tune BLIP VQA on the train split and evaluate on validation/test.

Outputs saved to `2_modeling/04_blip_finetune/results/`.
Heavy artifacts (checkpoints + final model) go to `2_modeling/04_blip_finetune/out/`.


In [2]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import BlipProcessor, BlipForQuestionAnswering
from transformers import TrainingArguments, Trainer
import evaluate


2026-02-05 21:11:17.653304: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-05 21:11:17.653393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-05 21:11:17.655663: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-05 21:11:17.664893: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 21:11:18.950208: W tensorflow/compiler/tf2

In [3]:
# Paths & config

def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. "
        "Run this notebook from within the Kvasir_VQA_x1 folder, "
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
RESULTS_DIR = DATA_ROOT / "2_modeling" / "04_blip_finetune" / "results"
OUT_DIR = DATA_ROOT / "2_modeling" / "04_blip_finetune" / "out"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = OUT_DIR / "checkpoints"
FINAL_MODEL_DIR = OUT_DIR / "final_model"

if not META_CSV.exists():
    raise RuntimeError(f"Missing metadata CSV: {META_CSV}")



# Model/training config
MODEL_NAME = "Salesforce/blip-vqa-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
QUESTION_MAX_LEN = 64
MAX_ANSWER_LEN = 16

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
LR = 1e-5

MAX_TRAIN_SAMPLES = None  # set int for smoke tests
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Train/val/test sample caps:", MAX_TRAIN_SAMPLES, MAX_VAL_SAMPLES, MAX_TEST_SAMPLES)
print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Results dir:", RESULTS_DIR)
print("Out dir:", OUT_DIR)


Model: Salesforce/blip-vqa-base
Device: cuda
Train/val/test sample caps: None None None
Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Results dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/04_blip_finetune/results
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/04_blip_finetune/out


In [4]:
# Load metadata
meta = pd.read_csv(META_CSV)

images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 143594, 'val': 0, 'test': 15955}


In [5]:
# Optional subsampling
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES is not None:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
if MAX_TEST_SAMPLES is not None:
    test_df = test_df.sample(min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 143594, 'val': 0, 'test': 15955}


In [6]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(DEVICE)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
model.config.pad_token_id = processor.tokenizer.pad_token_id
processor.tokenizer.padding_side = "right"

model.gradient_checkpointing_enable()
model.config.use_cache = False
if hasattr(model.config, "text_config"):
    model.config.text_config.pad_token_id = processor.tokenizer.pad_token_id


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [7]:
class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        question = str(row["question"])
        answer = str(row["answer"]) if pd.notna(row["answer"]) else ""
        inputs = processor(images=image, text=question, return_tensors="pt", padding="max_length", truncation=True, max_length=QUESTION_MAX_LEN)
        labels = processor.tokenizer(
            answer,
            max_length=MAX_ANSWER_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        decoder_input_ids = labels.clone()
        decoder_attention_mask = decoder_input_ids != processor.tokenizer.pad_token_id
        labels[decoder_input_ids == processor.tokenizer.pad_token_id] = -100
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["decoder_input_ids"] = decoder_input_ids
        item["decoder_attention_mask"] = decoder_attention_mask
        item["labels"] = labels
        return item
def collate_fn(batch):
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}


In [8]:
train_ds = VQADataset(train_df)
val_ds = VQADataset(val_df)
test_ds = VQADataset(test_df)


In [9]:
# Training args (compat across transformers versions)
import inspect

ta_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=[],
)

sig = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in sig.parameters:
    ta_kwargs["eval_strategy"] = "steps"
else:
    ta_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**ta_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)


In [ ]:
# Train
trainer.train()

# Save final model (heavy artifacts go to out/)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
print("Saved model to", FINAL_MODEL_DIR)


Step,Training Loss,Validation Loss
200,2.383000,No log
400,1.708800,No log
600,1.304700,No log
800,1.238300,No log
1000,1.096100,No log
1200,0.974400,No log
1400,0.928200,No log
1600,0.844900,No log
1800,0.834400,No log
2000,0.807500,No log


In [ ]:
# Evaluate (sampled) with BLEU/ROUGE
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

model.eval()

preds = []
refs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="BLIP eval"):
    img = Image.open(row["image_path"]).convert("RGB")
    q = str(row["question"])
    inputs = processor(images=img, text=q, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20)
    ans = processor.decode(out[0], skip_special_tokens=True)
    preds.append(ans)
    refs.append(str(row["answer"]))

bleu_refs = [[r] for r in refs]
bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"]

print("BLEU:", bleu_score)
print("ROUGE-L:", rouge_l)

metrics = {
    "bleu": bleu_score,
    "rougeL": rouge_l,
    "model": MODEL_NAME,
    "split": "test",
    "num_examples": len(test_df),
}
metrics_path = RESULTS_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote", metrics_path)

pred_path = RESULTS_DIR / "predictions.jsonl"
out_df = test_df.copy()
out_df["pred"] = preds
out_df.to_json(pred_path, orient="records", lines=True)
print("Wrote", pred_path)


NameError: name 'evaluate' is not defined